# 📧 Spam Email & SMS Detector (TF-IDF + Logistic Regression)

Welcome to the **Spam Email Detector** notebook! This project builds an end-to-end Machine Learning model to detect whether an incoming message or email is **SPAM** (unwanted / phishing / promotional scam) or **HAM** (legitimate communication).

### 🛠️ What We Will Do Step-by-Step:
1. **Import Libraries**: Load essential data science tools (`pandas`, `numpy`, `scikit-learn`, `joblib`).
2. **Load Datasets**: Load real email data (**Enron Spam Dataset**) and short text data (**SMS Spam Dataset**).
3. **Data Cleaning & Preprocessing**: Clean missing values, remove duplicates, and normalize labels.
4. **Train / Test Split**: Split data into 80% training and 20% testing using stratification.
5. **TF-IDF Vectorization**: Convert text words & phrases (unigrams + bigrams) into numerical importance weights.
6. **Model Training**: Train a **Logistic Regression** classifier from scratch.
7. **Evaluation**: Assess performance using Accuracy, Precision, Recall, F1-score, and Confusion Matrix.
8. **Custom Predictions**: Test the trained model with new emails and messages.
9. **Save Artifacts**: Export the model and vectorizer for production deployment.

## 1. Import Dependencies
We import pandas for data handling, scikit-learn for machine learning, and joblib for saving model artifacts.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import urllib.request, zipfile, os

# Set random seed for reproducibility
RANDOM_STATE = 42
print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Datasets
We define a robust `load_dataset` function that automatically loads or downloads the **Enron Email Dataset** and handles SMS data.

In [2]:
def load_dataset(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        if "enron" in path.name.lower():
            print(f"Downloading {path.name}...")
            url = "https://raw.githubusercontent.com/MWiechmann/enron_spam_data/master/enron_spam_data.zip"
            zip_target = path.parent / "enron_temp.zip"
            path.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, zip_target)
            with zipfile.ZipFile(zip_target) as z:
                z.extractall(path.parent)
            if zip_target.exists():
                os.remove(zip_target)
            print("Download complete!")
        else:
            return pd.DataFrame(columns=["label", "message"])
            
    try:
        df_raw = pd.read_csv(path)
        col_map = {str(c).lower().strip(): c for c in df_raw.columns}
        label_col = next((col_map[c] for c in ["spam", "label", "label_num", "spam/ham"] if c in col_map), None)
        has_subject = "subject" in col_map
        has_body = "body" in col_map or "message" in col_map
        text_col = next((col_map[c] for c in ["text", "message", "body", "content"] if c in col_map), None)
        
        if label_col and (text_col or (has_subject and has_body)):
            if has_subject and "message" in col_map and col_map["subject"] != col_map["message"]:
                sub = df_raw[col_map["subject"]].fillna("").astype(str)
                msg = df_raw[col_map["message"]].fillna("").astype(str)
                messages = "Subject: " + sub + "\n\n" + msg
            elif text_col:
                messages = df_raw[text_col].astype(str)
            else:
                messages = df_raw[col_map["body"]].astype(str)
            labels = df_raw[label_col].apply(lambda val: "spam" if str(val).strip().lower() in ["1", "spam", "true"] else "ham")
            return pd.DataFrame({"label": labels, "message": messages})
    except Exception:
        pass
        
    # Fallback to SMS tab-separated reader
    labels, messages = [], []
    with open(path, encoding="utf-8", errors="replace") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")
            if line.startswith('"') and line.endswith('"'): line = line[1:-1]
            if "\t" in line:
                l, m = line.split("\t", 1)
                labels.append(l.strip().lower())
                messages.append(m.strip())
    return pd.DataFrame({"label": labels, "message": messages})

dataset_dir = Path("dataset")
df = load_dataset(dataset_dir / "enron_spam_data.csv")
if (dataset_dir / "spam_dataset.csv").exists():
    df_sms = load_dataset(dataset_dir / "spam_dataset.csv")
    df = pd.concat([df, df_sms], ignore_index=True)

print(f"Total Combined Rows Loaded: {len(df)}")

Total Combined Rows Loaded: 39290


## 3. Dataset Inspection & Cleaning
Let's look at the dataset structure, missing values, duplicates, and class balance.

In [3]:
print("Shape (rows, columns):", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

# Cleaning missing values and duplicates
df_clean = df.dropna(subset=["label", "message"]).drop_duplicates().reset_index(drop=True)
print(f"\nRows after deduplication: {len(df_clean)}")

print("\nClass Distribution:")
print(df_clean["label"].value_counts())
print("\nClass Distribution (%):")
print(df_clean["label"].value_counts(normalize=True) * 100)

Shape (rows, columns): (39290, 2)

First 5 rows:


,label,message
0,ham,Subject: christmas tree farm pictures\n\n
1,ham,"Subject: vastar resources , inc .\n\ngary , pr..."
2,ham,Subject: calpine daily gas nomination\n\n- cal...
3,ham,Subject: re : issue\n\nfyi - see note below - ...
4,ham,Subject: meter 7268 nov allocation\n\nfyi .\n-...



Missing values:
label      0
message    0
dtype: int64

Rows after deduplication: 35654

Class Distribution:
label
ham     20428
spam    15226
Name: count, dtype: int64

Class Distribution (%):
label
ham     57.295114
spam    42.704886
Name: proportion, dtype: float64


## 4. Train / Test Split (Stratified 80/20)
We split our data into 80% training and 20% test data with stratification.

In [4]:
X = df_clean["message"]
y = df_clean["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

Training samples: 28523
Testing samples:  7131


## 5. TF-IDF Feature Extraction (Unigrams + Bigrams)
Extract 10,000 word and 2-word phrase features from the text.

In [5]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=10000,
    lowercase=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape:  {X_test_tfidf.shape}")

Vocabulary size: 10000
Train matrix shape: (28523, 10000)
Test matrix shape:  (7131, 10000)


## 6. Model Training (Logistic Regression)
Train the classifier on the TF-IDF feature matrix.

In [6]:
model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)
print("✅ Logistic Regression Model Trained Successfully!")

✅ Logistic Regression Model Trained Successfully!


## 7. Model Evaluation on Unseen Test Data
Compute accuracy, classification metrics, and confusion matrix.

In [7]:
y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=["ham", "spam"])

print("=" * 60)
print(f"MODEL ACCURACY: {acc:.4f} ({acc*100:.2f}%)")
print("=" * 60)
print("\nClassification Report:")
print(report)

print("Confusion Matrix (Rows = Actual, Columns = Predicted) [HAM, SPAM]:")
print(cm)
print(f"\n  True Negatives  (Ham correctly identified)  : {cm[0][0]}")
print(f"  False Positives (Ham wrongly flagged as spam): {cm[0][1]}")
print(f"  False Negatives (Spam that slipped through)  : {cm[1][0]}")
print(f"  True Positives  (Spam correctly caught)      : {cm[1][1]}")

MODEL ACCURACY: 0.9760 (97.60%)

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      0.98      0.98      4086
        spam       0.98      0.97      0.97      3045

    accuracy                           0.98      7131
   macro avg       0.98      0.98      0.98      7131
weighted avg       0.98      0.98      0.98      7131

Confusion Matrix (Rows = Actual, Columns = Predicted) [HAM, SPAM]:
[[4011   75]
 [  96 2949]]

  True Negatives  (Ham correctly identified)  : 4011
  False Positives (Ham wrongly flagged as spam): 75
  False Negatives (Spam that slipped through)  : 96
  True Positives  (Spam correctly caught)      : 2949


## 8. Interactive Predictions
Test the trained model on fresh real-world email and SMS examples.

In [17]:
def predict_email(text: str):
    text_tfidf = vectorizer.transform([text])
    prediction = model.predict(text_tfidf)[0]
    proba = model.predict_proba(text_tfidf)[0]

    spam_prob = proba[list(model.classes_).index("spam")]

    return prediction, spam_prob


sample_emails = [
    "URGENT: Your account access has been suspended. Click here to verify your identity immediately.",

    "FREE ENTRY! Win a £1000 cash prize or a brand new car! Text CLAIM to 87066 now!",

    "Congratulations! Your email was selected during our promotional drawing. You have qualified for a special cash reward.",

    "Nobody can decide where to eat and dad wants Chinese food tonight.",

    "Congratulations! You won $1,000,000. Claim your prize immediately.",

    "Your package is currently awaiting confirmation. Please verify your delivery details to complete the shipment."
]


print("=" * 80)
print(" SAMPLE PREDICTIONS")
print("=" * 80)

for email in sample_emails:
    pred, prob = predict_email(email)

    tag = "[🚨 SPAM]" if pred == "spam" else "[✅ HAM] "

    print(f"{tag} (Spam Prob: {prob:5.1%}) | {email}")


 SAMPLE PREDICTIONS
[🚨 SPAM] (Spam Prob: 88.0%) | URGENT: Your account access has been suspended. Click here to verify your identity immediately.
[🚨 SPAM] (Spam Prob: 96.7%) | FREE ENTRY! Win a £1000 cash prize or a brand new car! Text CLAIM to 87066 now!
[🚨 SPAM] (Spam Prob: 60.6%) | Congratulations! Your email was selected during our promotional drawing. You have qualified for a special cash reward.
[✅ HAM]  (Spam Prob: 12.3%) | Nobody can decide where to eat and dad wants Chinese food tonight.
[🚨 SPAM] (Spam Prob: 82.5%) | Congratulations! You won $1,000,000. Claim your prize immediately.
[✅ HAM]  (Spam Prob: 35.9%) | Your package is currently awaiting confirmation. Please verify your delivery details to complete the shipment.


## 9. Save Model & Vectorizer Artifacts
Export artifacts for production.

In [9]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

model_file = models_dir / "spam_classifier.joblib"
vec_file = models_dir / "tfidf_vectorizer.joblib"

joblib.dump(model, model_file)
joblib.dump(vectorizer, vec_file)

print(f"Model saved to:      {model_file}")
print(f"Vectorizer saved to: {vec_file}")
print("\n🎉 Project Complete!")

Model saved to:      models\spam_classifier.joblib
Vectorizer saved to: models\tfidf_vectorizer.joblib

🎉 Project Complete!
